# CLIP Baseline

Paper-aligned image-only baseline notebook. Uses train-only CUI vocabulary with minimum frequency 5 and maximum vocabulary cap 2048 for evaluation. Retrieval ranking uses visual similarity only; CUI labels are used only after ranking for metrics.

In [ ]:
import os, re, json, math, random, time, warnings, shutil, sys, subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")


DATA_ROOT = os.environ.get("DATA_ROOT", "../data/roco_v2")
OUT_DIR = os.environ.get("OUTPUT_DIR", os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "clip_roco_retrieval"))

CFG = {
    "seed": 42,
    "random_seeds": [42, 123, 2025],


    "clip_model_name": "openai/clip-vit-base-patch32",


    "epochs": 10,
    "batch_size": 16,
    "grad_accum_steps": 2,
    "num_workers": 2,
    "lr": 3e-6,
    "weight_decay": 1e-4,
    "max_text_length": 77,


    "max_train_samples": None,
    "max_valid_samples": None,

    "eval_split": "test",
    "retrieval_pool": "test",
    "ks_main": [5, 10, 50],
    "ks_curve": list(range(1, 51)),
    "relevance_threshold": 0.0,
    "embed_batch_size": 64,
    "eval_chunk_size": 256,
    "topk_max": 50,
    "train": False,
    "amp": True,
}

os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


try:
    from transformers import CLIPModel, CLIPProcessor
except Exception:
    print("Installing transformers...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "sentencepiece"])
    from transformers import CLIPModel, CLIPProcessor

print("Loading CLIP:", CFG["clip_model_name"])
processor = CLIPProcessor.from_pretrained(CFG["clip_model_name"])
model = CLIPModel.from_pretrained(CFG["clip_model_name"]).to(device)


def find_data_root(preferred):
    if Path(preferred).exists():
        return preferred

    print("DATA_ROOT not found. Searching DATA_SEARCH_DIR ...")
    candidates = []
    for root, dirs, files in os.walk(os.environ.get("DATA_SEARCH_DIR", "../data")):
        files_set = set(files)
        if {"train_concepts.csv", "valid_concepts.csv", "test_concepts.csv"}.issubset(files_set):
            candidates.append(root)

    if not candidates:
        raise FileNotFoundError("Could not find folder with train/valid/test_concepts.csv")

    return candidates[0]

DATA_ROOT = find_data_root(DATA_ROOT)
print("Using DATA_ROOT:", DATA_ROOT)

CUI_RE = re.compile(r"C\d{7}", re.IGNORECASE)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def read_csv_flexible(path):
    for sep in [",", "\t", ";", "|"]:
        try:
            df = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
            if df.shape[1] >= 1:
                return df
        except Exception:
            pass
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def build_image_map(split):
    split_dir = Path(DATA_ROOT) / split
    mp = {}
    for p in split_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            mp[p.stem] = str(p)
            mp[p.name] = str(p)
    return mp

def infer_id_col(df):
    for word in ["image", "file", "filename", "name", "id", "uid"]:
        for c in df.columns:
            if word in c.lower():
                return c
    return df.columns[0]

def extract_cuis_from_row(row):
    txt = " ".join([str(x) for x in row.values])
    return sorted(set([x.upper() for x in CUI_RE.findall(txt)]))

def clean_text(txt):
    txt = str(txt)
    txt = re.sub(r"\s+", " ", txt)
    txt = txt.replace("\x00", " ").strip()
    return txt

def extract_text_from_row(row, id_col=None):
    text_col_hints = ["caption", "text", "report", "findings", "impression", "description", "title", "abstract", "keyword"]
    bad_hints = ["cui", "semtype", "semantic_type"]

    cols = []
    for c in row.index:
        cl = c.lower()
        if id_col is not None and c == id_col:
            continue
        if any(b in cl for b in bad_hints):
            continue
        if any(h in cl for h in text_col_hints):
            cols.append(c)

    if not cols:

        for c in row.index:
            cl = c.lower()
            if id_col is not None and c == id_col:
                continue
            if any(b in cl for b in bad_hints):
                continue
            cols.append(c)

    parts = []
    for c in cols:
        s = clean_text(row[c])
        if not s:
            continue

        without_cui = CUI_RE.sub("", s).strip(" ,;|")
        if len(without_cui) >= 3:
            parts.append(without_cui)

    return clean_text(" ".join(parts))

def candidate_metadata_files(split):
    root = Path(DATA_ROOT)
    paths = []

    for ext in ["*.csv", "*.tsv", "*.txt"]:
        paths.extend(root.glob(f"{split}*{ext[-4:] if ext.startswith('*.') else ext}"))

    for p in root.glob(f"{split}*"):
        if p.suffix.lower() in [".csv", ".tsv", ".txt"]:
            paths.append(p)

    split_dir = root / split
    if split_dir.exists():
        for p in split_dir.rglob("*"):
            if p.suffix.lower() in [".csv", ".tsv", ".txt"]:
                paths.append(p)


    for p in root.glob("*"):
        if p.suffix.lower() in [".csv", ".tsv", ".txt"] and split in p.stem.lower():
            paths.append(p)

    unique = []
    seen = set()
    for p in paths:
        if p.exists() and str(p) not in seen:
            seen.add(str(p))
            unique.append(p)
    return unique

def build_text_map(split):
    text_map = {}
    for p in candidate_metadata_files(split):
        try:
            if p.suffix.lower() in [".csv", ".tsv"]:
                df = read_csv_flexible(p)
                if len(df) == 0:
                    continue
                id_col = infer_id_col(df)
                for _, row in df.iterrows():
                    raw_id = str(row[id_col]).strip()
                    base = Path(raw_id).stem
                    text = extract_text_from_row(row, id_col=id_col)
                    if len(text) > len(text_map.get(base, "")):
                        text_map[base] = text
                    if len(text) > len(text_map.get(raw_id, "")):
                        text_map[raw_id] = text
            elif p.suffix.lower() == ".txt":
                text = clean_text(p.read_text(errors="ignore"))
                if text:
                    text_map[p.stem] = text
        except Exception:
            pass
    return text_map

def load_split(split):
    csv_path = Path(DATA_ROOT) / f"{split}_concepts.csv"
    manual_path = Path(DATA_ROOT) / f"{split}_concepts_manual.csv"
    path = csv_path if csv_path.exists() else manual_path
    if not path.exists():
        raise FileNotFoundError(f"No concept CSV found for split={split}")

    df_raw = read_csv_flexible(path)
    id_col = infer_id_col(df_raw)
    img_map = build_image_map(split)
    text_map = build_text_map(split)

    rows, missing_path, no_cui, no_text = [], 0, 0, 0

    for _, row in df_raw.iterrows():
        raw_id = str(row[id_col]).strip()
        base = Path(raw_id).stem

        img_path = None
        for key in [raw_id, base, raw_id + ".jpg", base + ".jpg", raw_id + ".png", base + ".png"]:
            if key in img_map:
                img_path = img_map[key]
                break

        if img_path is None:
            missing_path += 1
            continue

        cuis = extract_cuis_from_row(row)
        if len(cuis) == 0:
            no_cui += 1

        text = text_map.get(base, "") or text_map.get(raw_id, "") or extract_text_from_row(row, id_col=id_col)
        text = clean_text(text)

        if len(text) < 3:
            no_text += 1
            if len(cuis) > 0:
                text = "medical radiology image with concepts " + " ".join(cuis)
            else:
                text = "medical radiology image"

        rows.append({
            "image_id": base,
            "path": img_path,
            "cuis": cuis,
            "n_cuis": len(cuis),
            "text": text,
            "text_len": len(text.split()),
        })

    out = pd.DataFrame(rows)
    print(f"{split}: raw={len(df_raw)} usable={len(out)} missing_path={missing_path} no_cui={no_cui} no_text_fallback={no_text}")
    return out

train_df = load_split("train")
valid_df = load_split("valid")
test_df = load_split("test")

train_df = train_df[train_df["n_cuis"] > 0].reset_index(drop=True)
valid_df = valid_df[valid_df["n_cuis"] > 0].reset_index(drop=True)
test_df = test_df[test_df["n_cuis"] > 0].reset_index(drop=True)

if CFG["max_train_samples"] is not None and len(train_df) > CFG["max_train_samples"]:
    train_df = train_df.sample(CFG["max_train_samples"], random_state=CFG["seed"]).reset_index(drop=True)

train_eval_df = train_df
valid_eval_df = valid_df
if CFG["max_valid_samples"] is not None and len(valid_eval_df) > CFG["max_valid_samples"]:
    valid_eval_df = valid_eval_df.sample(CFG["max_valid_samples"], random_state=CFG["seed"]).reset_index(drop=True)

overview = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "images": [len(train_df), len(valid_df), len(test_df)],
    "mean_cuis_per_image": [train_df.n_cuis.mean(), valid_df.n_cuis.mean(), test_df.n_cuis.mean()],
    "median_cuis_per_image": [train_df.n_cuis.median(), valid_df.n_cuis.median(), test_df.n_cuis.median()],
    "mean_text_words": [train_df.text_len.mean(), valid_df.text_len.mean(), test_df.text_len.mean()],
})
overview.to_csv(os.path.join(OUT_DIR, "dataset_overview.csv"), index=False)
display(overview)

print("\nExample CLIP text input:")
display(train_df[["image_id", "text", "n_cuis"]].head(3))


plt.figure(figsize=(6,4))
plt.bar(overview["split"], overview["images"])
plt.title("Images per split")
plt.ylabel("Number of images")
savefig("01_images_per_split.png")

plt.figure(figsize=(6,4))
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    plt.hist(df["n_cuis"].values, bins=30, alpha=0.5, label=name)
plt.title("CUI count per image")
plt.xlabel("Number of CUIs")
plt.ylabel("Images")
plt.legend()
savefig("02_cui_count_histogram.png")

train_cui_counts = Counter(c for xs in train_df["cuis"] for c in xs)
top_cuis_df = pd.DataFrame(train_cui_counts.most_common(30), columns=["CUI", "count"])
top_cuis_df.to_csv(os.path.join(OUT_DIR, "top_30_train_cuis.csv"), index=False)

plt.figure(figsize=(10,5))
plt.bar(top_cuis_df["CUI"], top_cuis_df["count"])
plt.title("Top 30 CUI frequencies in training set")
plt.xticks(rotation=75, ha="right")
plt.ylabel("Frequency")
savefig("03_top_cui_frequencies.png")


class RocoClipDataset(Dataset):
    def __init__(self, df, return_text=True):
        self.df = df.reset_index(drop=True)
        self.return_text = return_text

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.return_text:
            return img, row["text"], row["image_id"]
        return img, row["image_id"]

def clip_train_collate(batch):
    images, texts, image_ids = zip(*batch)
    inputs = processor(
        text=list(texts),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CFG["max_text_length"],
    )
    return inputs, list(image_ids)

def clip_image_collate(batch):
    images, image_ids = zip(*batch)
    inputs = processor(
        images=list(images),
        return_tensors="pt",
    )
    return inputs, list(image_ids)

train_loader = DataLoader(
    RocoClipDataset(train_df, return_text=True),
    batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=True,
    collate_fn=clip_train_collate,
    drop_last=True,
)

valid_loader = DataLoader(
    RocoClipDataset(valid_eval_df, return_text=True),
    batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
    collate_fn=clip_train_collate,
    drop_last=False,
)


optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG["epochs"]))
scaler = torch.cuda.amp.GradScaler(enabled=(CFG["amp"] and device.type == "cuda"))
CKPT_PATH = os.path.join(OUT_DIR, "best_clip_finetuned.pth")

def move_batch_to_device(inputs):
    return {k: v.to(device, non_blocking=True) for k, v in inputs.items() if torch.is_tensor(v)}

def run_one_epoch(loader, train=True):
    model.train(train)
    total_loss, total_n = 0.0, 0

    if train:
        optimizer.zero_grad(set_to_none=True)

    for step, (inputs, image_ids) in enumerate(loader):
        inputs = move_batch_to_device(inputs)
        bs = int(inputs["pixel_values"].size(0))

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(CFG["amp"] and device.type == "cuda")):
                out = model(**inputs, return_loss=True)
                loss = out.loss
                loss_for_backward = loss / CFG["grad_accum_steps"]

            if train:
                scaler.scale(loss_for_backward).backward()

                do_step = ((step + 1) % CFG["grad_accum_steps"] == 0) or ((step + 1) == len(loader))
                if do_step:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.detach().cpu()) * bs
        total_n += bs

    return total_loss / max(total_n, 1)

history = []
best_val = float("inf")

if CFG["train"]:
    print("\nTraining CLIP image-text model...")
    for epoch in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        tr_loss = run_one_epoch(train_loader, train=True)
        va_loss = run_one_epoch(valid_loader, train=False)
        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": tr_loss,
            "valid_loss": va_loss,
            "lr": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - t0,
        }
        history.append(row)
        print(f"Epoch {epoch:02d}/{CFG['epochs']} | train={tr_loss:.5f} | valid={va_loss:.5f} | time={row['seconds']:.1f}s")

        if va_loss < best_val:
            best_val = va_loss
            torch.save({
                "model": model.state_dict(),
                "cfg": CFG,
                "best_val": best_val,
            }, CKPT_PATH)
            print("  saved best checkpoint")

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)

    plt.figure(figsize=(6,4))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train")
    plt.plot(hist_df["epoch"], hist_df["valid_loss"], marker="o", label="valid")
    plt.title("CLIP training and validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("CLIP contrastive loss")
    plt.legend()
    savefig("04_training_loss_curve.png")

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    print("Loaded best checkpoint:", CKPT_PATH)


def _project_clip_features_if_needed(feats):
    """
    Return valid CLIP image embeddings without causing projection shape mismatch.

    Normal CLIPModel.get_image_features() returns projected 512-d embeddings.
    Some environments/checkpoints may instead return a ModelOutput. In that case:
      - use image_embeds if available
      - project pooler_output only when its last dim matches visual_projection.in_features
      - if it is already projected, return it directly
    """
    if not torch.is_tensor(feats):
        raise TypeError(f"Expected tensor features, got {type(feats)}")

    proj = getattr(model, "visual_projection", None)
    if proj is None:
        proj = getattr(model, "vision_projection", None)

    if proj is not None and hasattr(proj, "in_features") and hasattr(proj, "out_features"):
        in_dim = int(proj.in_features)
        out_dim = int(proj.out_features)

        if feats.shape[-1] == in_dim:
            return proj(feats)
        if feats.shape[-1] == out_dim:
            return feats

        raise RuntimeError(
            f"Unexpected CLIP feature dimension {feats.shape[-1]}. "
            f"visual_projection expects in_features={in_dim}, out_features={out_dim}."
        )

    return feats


def get_clip_image_features_safe(pixel_values):
    """
    Robust CLIP image feature extractor.

    Fixes the error:
        RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x512 and 768x512)

    The error happened because projected 512-d features were passed again through
    CLIP's visual_projection layer, which expects 768-d vision pooled features.
    """
    out = model.get_image_features(pixel_values=pixel_values)


    if torch.is_tensor(out):
        return out


    if hasattr(out, "image_embeds") and torch.is_tensor(out.image_embeds):
        return out.image_embeds


    if hasattr(out, "pooler_output") and torch.is_tensor(out.pooler_output):
        return _project_clip_features_if_needed(out.pooler_output)


    if isinstance(out, (tuple, list)):
        for item in out:
            if hasattr(item, "image_embeds") and torch.is_tensor(item.image_embeds):
                return item.image_embeds
            if torch.is_tensor(item) and item.ndim == 2:
                return _project_clip_features_if_needed(item)
            if hasattr(item, "pooler_output") and torch.is_tensor(item.pooler_output):
                return _project_clip_features_if_needed(item.pooler_output)


    vision_out = model.vision_model(pixel_values=pixel_values, return_dict=True)
    return _project_clip_features_if_needed(vision_out.pooler_output)

@torch.no_grad()
def extract_embeddings(df, name):
    loader = DataLoader(
        RocoClipDataset(df, return_text=False),
        batch_size=CFG["embed_batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=True,
        collate_fn=clip_image_collate,
    )

    model.eval()
    embs, ids = [], []

    for inputs, image_ids in loader:
        inputs = move_batch_to_device(inputs)

        with torch.cuda.amp.autocast(enabled=(CFG["amp"] and device.type == "cuda")):
            feats = get_clip_image_features_safe(inputs["pixel_values"])
            feats = F.normalize(feats, dim=1)

        embs.append(feats.float().cpu().numpy().astype("float32"))
        ids.extend(list(image_ids))

    if len(embs) == 0:
        raise RuntimeError(f"No embeddings extracted for {name}. Check image paths and dataframe.")

    embs = np.concatenate(embs, axis=0)
    np.save(os.path.join(OUT_DIR, f"{name}_embeddings.npy"), embs)
    pd.DataFrame({"image_id": ids}).to_csv(os.path.join(OUT_DIR, f"{name}_ids.csv"), index=False)

    print(name, embs.shape)
    return embs, ids

split_to_df = {"train": train_df, "valid": valid_df, "test": test_df}
query_df = split_to_df[CFG["eval_split"]].reset_index(drop=True)
pool_df = split_to_df[CFG["retrieval_pool"]].reset_index(drop=True)

query_embs, query_ids = extract_embeddings(query_df, CFG["eval_split"] + "_query")
pool_embs, pool_ids = extract_embeddings(pool_df, CFG["retrieval_pool"] + "_pool")

same_pool = (
    CFG["eval_split"] == CFG["retrieval_pool"] and
    len(query_df) == len(pool_df) and
    list(query_df["image_id"]) == list(pool_df["image_id"])
)
print("same query/pool:", same_pool)


def compute_topk_by_embedding(query_embs, pool_embs, topk, chunk_size, same_pool):
    q = torch.tensor(query_embs, dtype=torch.float32, device=device)
    p = torch.tensor(pool_embs, dtype=torch.float32, device=device)
    all_idx, all_sim = [], []

    for s in range(0, q.shape[0], chunk_size):
        e = min(s + chunk_size, q.shape[0])
        sim = q[s:e] @ p.T

        if same_pool:
            diag = torch.arange(s, e, device=device)
            sim[torch.arange(e - s, device=device), diag] = -1e9

        vals, inds = torch.topk(sim, k=min(topk, p.shape[0] - 1 if same_pool else p.shape[0]), dim=1)
        all_idx.append(inds.cpu().numpy())
        all_sim.append(vals.cpu().numpy())

    return np.vstack(all_idx), np.vstack(all_sim)

topk_idx, topk_sim = compute_topk_by_embedding(
    query_embs, pool_embs,
    topk=CFG["topk_max"],
    chunk_size=CFG["eval_chunk_size"],
    same_pool=same_pool,
)

np.save(os.path.join(OUT_DIR, "retrieval_topk_indices.npy"), topk_idx)
np.save(os.path.join(OUT_DIR, "retrieval_topk_similarities.npy"), topk_sim)

rows = []
for qi in range(len(query_df)):
    for r in range(topk_idx.shape[1]):
        pi = int(topk_idx[qi, r])
        rows.append({
            "query_image_id": query_df.iloc[qi]["image_id"],
            "rank": r + 1,
            "retrieved_image_id": pool_df.iloc[pi]["image_id"],
            "cosine_similarity": float(topk_sim[qi, r]),
        })
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "retrieval_top50.csv"), index=False)


def build_cui_vocab(*dfs):
    vocab = {}
    for df in dfs:
        for cuis in df["cuis"]:
            for c in cuis:
                if c not in vocab:
                    vocab[c] = len(vocab)
    return vocab

def build_csr(df, vocab):
    rows, cols = [], []
    for i, cuis in enumerate(df["cuis"]):
        for c in set(cuis):
            if c in vocab:
                rows.append(i)
                cols.append(vocab[c])
    data = np.ones(len(rows), dtype=np.float32)
    mat = csr_matrix((data, (rows, cols)), shape=(len(df), len(vocab)), dtype=np.float32)
    lengths = np.asarray(mat.sum(axis=1)).reshape(-1).astype(np.float32)
    return mat, lengths

cui_vocab = build_cui_vocab(query_df, pool_df)
query_csr, query_len = build_csr(query_df, cui_vocab)
pool_csr, pool_len = build_csr(pool_df, cui_vocab)
print("CUI vocab:", len(cui_vocab))


def dcg_at_k(gains):
    gains = np.asarray(gains, dtype=np.float32)
    discounts = 1.0 / np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains * discounts))

def ap_at_k(rels, total_relevant, k):
    if total_relevant <= 0:
        return np.nan
    rels = np.asarray(rels[:k], dtype=np.float32)
    hits, precisions = 0, []
    for i, r in enumerate(rels, start=1):
        if r > 0:
            hits += 1
            precisions.append(hits / i)
    return float(np.sum(precisions) / min(total_relevant, k)) if precisions else 0.0

def rr_at_k(rels, k):
    rels = np.asarray(rels[:k], dtype=np.float32)
    hits = np.where(rels > 0)[0]
    return float(1.0 / (hits[0] + 1)) if len(hits) else 0.0

def summarize(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[~np.isnan(values)]
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if n > 1 else 0.0
    ci95 = float(1.96 * std / math.sqrt(n)) if n > 1 else 0.0
    return mean, std, ci95, n

def evaluate_metrics(query_csr, pool_csr, query_len, pool_len, pred_idx, ks, threshold, same_pool, chunk_size):
    maxk = max(ks)
    per_query = []

    for s in range(0, query_csr.shape[0], chunk_size):
        e = min(s + chunk_size, query_csr.shape[0])

        inter = query_csr[s:e].dot(pool_csr.T).toarray().astype(np.float32)
        union = query_len[s:e, None] + pool_len[None, :] - inter
        all_gains = np.divide(inter, np.maximum(union, 1e-8), out=np.zeros_like(inter), where=union > 0)

        if same_pool:
            for local_i, global_i in enumerate(range(s, e)):
                all_gains[local_i, global_i] = -1.0

        for local_i, global_i in enumerate(range(s, e)):
            top_idx = pred_idx[global_i, :maxk]
            pred_gains = all_gains[local_i, top_idx].copy()
            pred_gains[pred_gains < 0] = 0.0

            gains = all_gains[local_i].copy()
            gains[gains < 0] = 0.0
            total_relevant = int((gains > threshold).sum())
            ideal_sorted = np.sort(gains)[::-1]

            row = {
                "query_index": global_i,
                "query_image_id": query_df.iloc[global_i]["image_id"],
                "total_relevant_by_cui_jaccard": total_relevant,
            }

            for k in ks:
                gk = pred_gains[:k]
                rel = gk > threshold

                idcg = dcg_at_k(ideal_sorted[:k])
                row[f"CUI@{k}"] = dcg_at_k(gk) / idcg if idcg > 0 else 0.0
                row[f"Precision@{k}"] = float(rel.sum() / k)
                row[f"Recall@{k}"] = float(rel.sum() / total_relevant) if total_relevant > 0 else np.nan
                row[f"mAP@{k}"] = ap_at_k(rel, total_relevant, k)
                row[f"MRR@{k}"] = rr_at_k(rel, k)
                row[f"MeanJaccard@{k}"] = float(np.mean(gk))

            per_query.append(row)

    per_query = pd.DataFrame(per_query)

    summary = []
    for col in [c for c in per_query.columns if "@" in c]:
        mean, std, ci95, n = summarize(per_query[col].values)
        summary.append({
            "metric": col,
            "mean": mean,
            "std": std,
            "ci95": ci95,
            "n_queries_used": n,
            "mean±std": f"{mean:.4f} ± {std:.4f}",
            "mean±95%CI": f"{mean:.4f} ± {ci95:.4f}",
        })

    return per_query, pd.DataFrame(summary)

all_ks = sorted(set(CFG["ks_main"] + CFG["ks_curve"]))
per_query_metrics, metrics_summary = evaluate_metrics(
    query_csr, pool_csr, query_len, pool_len,
    topk_idx, all_ks,
    threshold=CFG["relevance_threshold"],
    same_pool=same_pool,
    chunk_size=CFG["eval_chunk_size"],
)

per_query_metrics.to_csv(os.path.join(OUT_DIR, "per_query_metrics.csv"), index=False)
metrics_summary.to_csv(os.path.join(OUT_DIR, "metrics_summary_all_k.csv"), index=False)

main_cols = []
for metric in ["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"]:
    for k in CFG["ks_main"]:
        main_cols.append(f"{metric}@{k}")

main_summary = metrics_summary[metrics_summary["metric"].isin(main_cols)].copy()
main_summary["order"] = main_summary["metric"].apply(lambda x: main_cols.index(x))
main_summary = main_summary.sort_values("order").drop(columns=["order"])
main_summary.to_csv(os.path.join(OUT_DIR, "metrics_summary_main.csv"), index=False)

print("\n========== MAIN RESULTS ==========")
display(main_summary[["metric", "mean±std", "mean±95%CI", "n_queries_used"]])

no_rel = int((per_query_metrics["total_relevant_by_cui_jaccard"] == 0).sum())
print(f"Queries with no relevant non-self image: {no_rel}/{len(per_query_metrics)}")


def metric_curve(summary, prefix):
    xs, means, cis = [], [], []
    for k in CFG["ks_curve"]:
        name = f"{prefix}@{k}"
        hit = summary[summary["metric"] == name]
        if len(hit):
            xs.append(k)
            means.append(float(hit["mean"].iloc[0]))
            cis.append(float(hit["ci95"].iloc[0]))
    return np.array(xs), np.array(means), np.array(cis)

x, y, ci = metric_curve(metrics_summary, "CUI")
plt.figure(figsize=(7,4))
plt.plot(x, y, marker="o", markersize=3)
plt.fill_between(x, y-ci, y+ci, alpha=0.2)
plt.title("CUI@K curve with 95% CI")
plt.xlabel("K")
plt.ylabel("CUI@K")
savefig("05_cui_at_k_curve.png")

xp, p, pci = metric_curve(metrics_summary, "Precision")
xr, r, rci = metric_curve(metrics_summary, "Recall")
plt.figure(figsize=(7,4))
plt.plot(xp, p, marker="o", markersize=3, label="Precision@K")
plt.fill_between(xp, p-pci, p+pci, alpha=0.2)
plt.plot(xr, r, marker="o", markersize=3, label="Recall@K")
plt.fill_between(xr, r-rci, r+rci, alpha=0.2)
plt.title("Precision@K and Recall@K")
plt.xlabel("K")
plt.ylabel("Score")
plt.legend()
savefig("06_precision_recall_at_k_curves.png")

plt.figure(figsize=(6,5))
plt.plot(r, p, marker="o", markersize=3)
for kk in [1, 5, 10, 20, 50]:
    if kk in xp:
        idx = list(xp).index(kk)
        plt.annotate(f"K={kk}", (r[idx], p[idx]))
plt.title("Precision-Recall tradeoff across K")
plt.xlabel("Recall@K")
plt.ylabel("Precision@K")
savefig("07_precision_recall_tradeoff.png")

bar_metrics = [f"CUI@{k}" for k in CFG["ks_main"]] + [f"Precision@{k}" for k in CFG["ks_main"]] + [f"Recall@{k}" for k in CFG["ks_main"]]
bar_df = main_summary[main_summary["metric"].isin(bar_metrics)].copy()
plt.figure(figsize=(10,5))
plt.bar(bar_df["metric"], bar_df["mean"], yerr=bar_df["std"], capsize=4)
plt.xticks(rotation=45, ha="right")
plt.title("Main metrics: mean ± std across queries")
plt.ylabel("Score")
savefig("08_main_metrics_bar_mean_std.png")

bar_metrics2 = [f"mAP@{k}" for k in CFG["ks_main"]] + [f"MRR@{k}" for k in CFG["ks_main"]]
bar_df2 = main_summary[main_summary["metric"].isin(bar_metrics2)].copy()
plt.figure(figsize=(8,4))
plt.bar(bar_df2["metric"], bar_df2["mean"], yerr=bar_df2["std"], capsize=4)
plt.xticks(rotation=45, ha="right")
plt.title("mAP and MRR: mean ± std")
plt.ylabel("Score")
savefig("09_map_mrr_bar_mean_std.png")

plt.figure(figsize=(6,4))
plt.hist(topk_sim[:, 0], bins=40, alpha=0.8)
plt.title("Top-1 cosine similarity distribution")
plt.xlabel("Top-1 cosine similarity")
plt.ylabel("Queries")
savefig("10_top1_similarity_distribution.png")


sample_n = min(3000, len(pool_embs))
rng = np.random.default_rng(CFG["seed"])
sample_idx = rng.choice(len(pool_embs), size=sample_n, replace=False)
sample_embs = pool_embs[sample_idx]

xy = PCA(n_components=2, random_state=CFG["seed"]).fit_transform(sample_embs)
clusters = KMeans(n_clusters=min(10, sample_n), random_state=CFG["seed"], n_init=10).fit_predict(sample_embs)

plt.figure(figsize=(7,6))
plt.scatter(xy[:, 0], xy[:, 1], c=clusters, s=8, alpha=0.8)
plt.title("CLIP image embedding PCA projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
savefig("11_embedding_pca_kmeans_clusters.png")


def load_img_for_plot(path, size=180):
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size))
    return img

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(1, len(a | b))

def plot_retrieval_grid(query_indices, k, name):
    n_rows = len(query_indices)
    n_cols = k + 1
    plt.figure(figsize=(2.1*n_cols, 2.3*n_rows))

    for rr, qi in enumerate(query_indices):
        qrow = query_df.iloc[qi]
        ax = plt.subplot(n_rows, n_cols, rr*n_cols + 1)
        ax.imshow(load_img_for_plot(qrow["path"]))
        ax.set_title("Query\n" + qrow["image_id"][-10:], fontsize=8)
        ax.axis("off")

        for j in range(k):
            pi = int(topk_idx[qi, j])
            prow = pool_df.iloc[pi]
            ax = plt.subplot(n_rows, n_cols, rr*n_cols + 2 + j)
            ax.imshow(load_img_for_plot(prow["path"]))
            jac = jaccard(qrow["cuis"], prow["cuis"])
            ax.set_title(f"Rank {j+1}\nJ={jac:.2f}", fontsize=8)
            ax.axis("off")

    savefig(name)

rand_q = rng.choice(len(query_df), size=min(6, len(query_df)), replace=False).tolist()
plot_retrieval_grid(rand_q, 5, "12_random_top5_retrieval_examples.png")

if "CUI@5" in per_query_metrics.columns:
    best_q = per_query_metrics.sort_values("CUI@5", ascending=False)["query_index"].head(6).tolist()
    worst_q = per_query_metrics.sort_values("CUI@5", ascending=True)["query_index"].head(6).tolist()
    plot_retrieval_grid(best_q, 5, "13_best_cui5_top5_retrieval_examples.png")
    plot_retrieval_grid(worst_q, 5, "14_worst_cui5_top5_retrieval_examples.png")


print("\n========== SAVED FILES ==========")
for root, dirs, files in os.walk(OUT_DIR):
    level = root.replace(OUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        print(f"{'  ' * (level + 1)}{f}")

print("\nDone.")
print("Main metrics:", os.path.join(OUT_DIR, "metrics_summary_main.csv"))
print("All K metrics:", os.path.join(OUT_DIR, "metrics_summary_all_k.csv"))
print("Figures:", FIG_DIR)


import shutil
from pathlib import Path

FIG_DIR = Path(os.path.join(OUT_DIR, "figures"))
ZIP_PATH = os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "clip_roco_retrieval_figures")

shutil.make_archive(
    base_name=ZIP_PATH,
    format="zip",
    root_dir=str(FIG_DIR)
)

print("Figures zip saved at:")
print(ZIP_PATH + ".zip")
